In [1]:
# ==========================================
# CELL 1 - IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from ortools.linear_solver import pywraplp

print("All Libraries Imported Successfully!")

All Libraries Imported Successfully!


In [2]:
# ==========================================
# CELL 2 - LOAD ALL NESTLÉ DATASETS
# ==========================================

# Orders Dataset
orders = pd.read_csv("../data/orders_clean.csv")

# Inventory / Capacity Dataset
capacity = pd.read_csv("../data/input data/input_capacity_planning.csv")

# Shipping Cost Dataset
shipping = pd.read_csv("../data/input data/input_shipping_cost_data.csv")

# Dock Capacity Dataset
dock = pd.read_csv("../data/input data/input_dock_capacity.csv")

# Throughput Capacity Dataset
throughput = pd.read_csv("../data/input data/input_throughput_capacity.csv")

print("All Nestlé datasets loaded successfully!")

All Nestlé datasets loaded successfully!


In [3]:
# ==========================================
# CELL 3 - EXPLORE DATASETS
# ==========================================

print("=" * 60)
print("ORDERS DATASET")
print("=" * 60)
print("Shape :", orders.shape)
print("Columns:")
print(orders.columns.tolist())

print("\n" + "=" * 60)
print("CAPACITY DATASET")
print("=" * 60)
print("Shape :", capacity.shape)
print("Columns:")
print(capacity.columns.tolist())

print("\n" + "=" * 60)
print("SHIPPING DATASET")
print("=" * 60)
print("Shape :", shipping.shape)
print("Columns:")
print(shipping.columns.tolist())

print("\n" + "=" * 60)
print("DOCK DATASET")
print("=" * 60)
print("Shape :", dock.shape)
print("Columns:")
print(dock.columns.tolist())

print("\n" + "=" * 60)
print("THROUGHPUT DATASET")
print("=" * 60)
print("Shape :", throughput.shape)
print("Columns:")
print(throughput.columns.tolist())

print("\n✅ Dataset exploration completed successfully!")

ORDERS DATASET
Shape : (25193, 36)
Columns:
['Group_Flag', 'Plant', 'MaterialNumber', 'transportationplanningdate', 'IsTopCust', 'OpeningStock', 'RequestedDeliveryDate', 'DeliveryNoteFlag', 'IsInvAvail', 'LoadNumber', 'DeliveryPriority', 'ProductPlanningUnitOfMeasure', 'ProductPlanningUnitsPerCase', 'ProductPlanningUnitsPerPallet', 'OrderedQty_converted', 'OrderedWeight', 'OrderedVolume', 'Order_SKU_Revenue', 'ZipCode', 'ProductCasesPerPallet', 'CalculatedFootprints', 'IsCOF<100', 'calculated_ordered_weight', 'IsFTL', 'IsMultiplePlant', 'IsMultiplePGI', 'IsMultipleRDD', 'Measure', 'FillRateThreshold', 'Penaltyforpotentialcuts', 'MaximumPenalty', 'FixedPenalty', 'FixedPenaltyPerSKU', 'MinimumPenalty', 'OnTimePercentage', 'OnTimeFixed']

CAPACITY DATASET
Shape : (377504, 23)
Columns:
['LocationID', 'MaterialID', 'DATE', 'OpeningStock', 'SalesActualCustomerOrders', 'Total_Unreserved_Qty', 'ClosingStock', 'TotalSupply', 'BatchAvailability', 'OutgoingDispatchPlan', 'IncomingDispatchPlan', '

In [4]:
# ==========================================
# CELL 4 - PREPARE SAMPLE ORDERS
# ==========================================

# Work on a copy of the complete orders dataset
sample_orders = orders.copy()

# Convert important columns to correct data types
sample_orders["Plant"] = sample_orders["Plant"].astype(str)
sample_orders["MaterialNumber"] = sample_orders["MaterialNumber"].astype(str)

sample_orders["OrderedQty_converted"] = pd.to_numeric(
    sample_orders["OrderedQty_converted"],
    errors="coerce"
).fillna(0)

sample_orders["Order_SKU_Revenue"] = pd.to_numeric(
    sample_orders["Order_SKU_Revenue"],
    errors="coerce"
).fillna(0)

print("Sample Orders Prepared Successfully!")

print("\nShape:", sample_orders.shape)

print("\nFirst 5 Orders:")
print(sample_orders.head())

Sample Orders Prepared Successfully!

Shape: (25193, 36)

First 5 Orders:
   Group_Flag Plant MaterialNumber transportationplanningdate IsTopCust  \
0  5484913123  5083       12260382                    6/26/24         N   
1  5484913123  5083        9516458                    6/26/24         N   
2  5484913123  5083       12408924                    6/26/24         N   
3  5484913123  5083        9517630                    6/26/24         N   
4  5484913123  5083       12587091                    6/26/24         N   

   OpeningStock RequestedDeliveryDate DeliveryNoteFlag IsInvAvail  LoadNumber  \
0       33814.0               6/27/24                N          Y  U600105191   
1        6546.0               6/27/24                N          Y  U600105191   
2        4151.0               6/27/24                N          Y  U600105191   
3       17667.0               6/27/24                N          Y  U600105191   
4       89341.0               6/27/24                N          Y  U60

In [5]:
# ==========================================
# CELL 5 - MERGE INVENTORY
# ==========================================

# Remove old inventory column if it already exists
if "Available_inventory" in sample_orders.columns:
    sample_orders = sample_orders.drop(columns=["Available_inventory"])

# Prepare inventory table
capacity_small = capacity[
    ["LocationID", "MaterialID", "Available_inventory"]
].copy()

capacity_small = capacity_small.rename(columns={
    "LocationID": "Plant",
    "MaterialID": "MaterialNumber"
})

# Convert key columns to string
capacity_small["Plant"] = capacity_small["Plant"].astype(str)
capacity_small["MaterialNumber"] = capacity_small["MaterialNumber"].astype(str)

# Sum inventory for each Plant + Material
inventory = (
    capacity_small
    .groupby(["Plant", "MaterialNumber"], as_index=False)
    ["Available_inventory"]
    .sum()
)

# Merge into orders
sample_orders = sample_orders.merge(
    inventory,
    on=["Plant", "MaterialNumber"],
    how="left"
)

# Replace missing inventory with 0
sample_orders["Available_inventory"] = (
    sample_orders["Available_inventory"]
    .fillna(0)
)

print("Inventory merged successfully!")

print(sample_orders[
    ["Plant", "MaterialNumber", "Available_inventory"]
].head())

print("\nCurrent Shape:", sample_orders.shape)

Inventory merged successfully!
  Plant MaterialNumber  Available_inventory
0  5083       12260382             941546.0
1  5083        9516458             269679.0
2  5083       12408924             234719.0
3  5083        9517630             842948.0
4  5083       12587091            2428992.0

Current Shape: (25193, 37)


In [6]:
# ==========================================
# CELL 6 - MERGE SHIPPING COST
# ==========================================

# Remove old Shipping_Cost column if it already exists
if "Shipping_Cost" in sample_orders.columns:
    sample_orders = sample_orders.drop(columns=["Shipping_Cost"])

# Keep only required columns
shipping_small = shipping[
    ["Plant", "Shipping_Cost"]
].copy()

# Convert Plant to string
shipping_small["Plant"] = shipping_small["Plant"].astype(str)

# Average shipping cost for each plant
shipping_small = (
    shipping_small
    .groupby("Plant", as_index=False)
    .agg({
        "Shipping_Cost": "mean"
    })
)

# Merge into orders
sample_orders = sample_orders.merge(
    shipping_small,
    on="Plant",
    how="left"
)

# Fill missing values
sample_orders["Shipping_Cost"] = (
    sample_orders["Shipping_Cost"]
    .fillna(0)
)

print("Shipping cost merged successfully!")

print(
    sample_orders[
        ["Plant", "Shipping_Cost"]
    ].head()
)

print("\nCurrent Shape:", sample_orders.shape)

Shipping cost merged successfully!
  Plant  Shipping_Cost
0  5083    2076.747562
1  5083    2076.747562
2  5083    2076.747562
3  5083    2076.747562
4  5083    2076.747562

Current Shape: (25193, 38)


In [7]:
# ==========================================
# CELL 7 - MERGE DOCK CAPACITY
# ==========================================

# Remove old Dock_Remaining column if it exists
if "Dock_Remaining" in sample_orders.columns:
    sample_orders = sample_orders.drop(columns=["Dock_Remaining"])

# Keep required columns
dock_small = dock[
    ["Plant", "Dock_Remaining"]
].copy()

# Convert Plant to string
dock_small["Plant"] = dock_small["Plant"].astype(str)

# Average remaining dock capacity per plant
dock_small = (
    dock_small
    .groupby("Plant", as_index=False)
    .agg({
        "Dock_Remaining": "mean"
    })
)

# Merge into orders
sample_orders = sample_orders.merge(
    dock_small,
    on="Plant",
    how="left"
)

# Fill missing values
sample_orders["Dock_Remaining"] = (
    sample_orders["Dock_Remaining"]
    .fillna(0)
)

print("Dock capacity merged successfully!")

print(
    sample_orders[
        ["Plant", "Dock_Remaining"]
    ].head()
)

print("\nCurrent Shape:", sample_orders.shape)

Dock capacity merged successfully!
  Plant  Dock_Remaining
0  5083             0.0
1  5083             0.0
2  5083             0.0
3  5083             0.0
4  5083             0.0

Current Shape: (25193, 39)


In [8]:
# ==========================================================
# CELL 8 - Merge Throughput Capacity
# ==========================================================

# Keep only required columns
throughput_small = throughput[
    ["Plant", "order_count"]
].copy()

# Convert Plant to string
throughput_small["Plant"] = throughput_small["Plant"].astype(str)

# Calculate total throughput capacity per plant
throughput_summary = (
    throughput_small
    .groupby("Plant")["order_count"]
    .sum()
    .reset_index()
)

# Rename column
throughput_summary = throughput_summary.rename(
    columns={"order_count": "Throughput_Capacity"}
)

# Convert Plant in orders
sample_orders["Plant"] = sample_orders["Plant"].astype(str)

# Merge into orders
sample_orders = sample_orders.merge(
    throughput_summary,
    on="Plant",
    how="left"
)

# Replace missing values
sample_orders["Throughput_Capacity"] = (
    sample_orders["Throughput_Capacity"]
    .fillna(0)
)

print("Throughput capacity merged successfully!\n")

print(sample_orders[
    ["Plant", "Throughput_Capacity"]
].head())

print("\nCurrent Shape:", sample_orders.shape)

Throughput capacity merged successfully!

  Plant  Throughput_Capacity
0  5083                  957
1  5083                  957
2  5083                  957
3  5083                  957
4  5083                  957

Current Shape: (25193, 40)


In [9]:
# ==========================================================
# CELL 9 - Create OR-Tools Solver
# ==========================================================

from ortools.linear_solver import pywraplp

# Create SCIP Solver
solver = pywraplp.Solver.CreateSolver("SCIP")

if solver:
    print("OR-Tools SCIP Solver created successfully!")
else:
    print("Solver could not be created!")

# Display solver information
print("Solver Name :", solver.SolverVersion())

OR-Tools SCIP Solver created successfully!
Solver Name : SCIP 10.0.0 [LP solver: SoPlex 8.0.0]


In [10]:
# ==========================================================
# CELL 10 - Create Decision Variables
# ==========================================================

# Create one binary decision variable for every order
x = {}

for i in sample_orders.index:
    x[i] = solver.BoolVar(f"Order_{i}")

print("Decision Variables Created Successfully!")
print("Total Decision Variables :", len(x))

Decision Variables Created Successfully!
Total Decision Variables : 25193


In [11]:
# ==========================================================
# CELL 11 - Objective Function
# Maximize Revenue - Shipping Cost - Penalty
# ==========================================================

# Fill missing values
sample_orders["Shipping_Cost"] = sample_orders["Shipping_Cost"].fillna(0)
sample_orders["Penaltyforpotentialcuts"] = sample_orders["Penaltyforpotentialcuts"].fillna(0)
sample_orders["Order_SKU_Revenue"] = sample_orders["Order_SKU_Revenue"].fillna(0)

# Objective Function
solver.Maximize(

    solver.Sum(

        (
            sample_orders.loc[i, "Order_SKU_Revenue"]
            - sample_orders.loc[i, "Shipping_Cost"]
            - sample_orders.loc[i, "Penaltyforpotentialcuts"]
        ) * x[i]

        for i in sample_orders.index

    )

)

print("Objective Function Added Successfully!")
print("Objective = Revenue - Shipping Cost - Penalty")

Objective Function Added Successfully!
Objective = Revenue - Shipping Cost - Penalty


In [12]:
# ==========================================================
# CELL 12 - Inventory Constraints
# ==========================================================

inventory_constraints = 0

for i in sample_orders.index:

    if sample_orders.loc[i, "Available_inventory"] < sample_orders.loc[i, "OrderedQty_converted"]:
        solver.Add(x[i] == 0)
        inventory_constraints += 1

print("Inventory Constraints Added Successfully!")
print("Orders blocked due to insufficient inventory:", inventory_constraints)

Inventory Constraints Added Successfully!
Orders blocked due to insufficient inventory: 318


In [13]:
# ==========================================================
# CELL 13 - Plant Capacity Constraints
# ==========================================================

plant_constraints = 0

for plant in sample_orders["Plant"].unique():

    plant_orders = sample_orders[
        sample_orders["Plant"] == plant
    ].index.tolist()

    capacity_limit = sample_orders[
        sample_orders["Plant"] == plant
    ]["Throughput_Capacity"].iloc[0]

    solver.Add(
        solver.Sum(x[i] for i in plant_orders) <= capacity_limit
    )

    plant_constraints += 1

print("Plant Capacity Constraints Added Successfully!")
print("Total Plants:", plant_constraints)

Plant Capacity Constraints Added Successfully!
Total Plants: 8


In [14]:
# ==========================================================
# CELL 14 - Dock Capacity Constraints
# ==========================================================

dock_constraints = 0

for plant in sample_orders["Plant"].unique():

    plant_orders = sample_orders[
        sample_orders["Plant"] == plant
    ].index.tolist()

    dock_limit = sample_orders[
        sample_orders["Plant"] == plant
    ]["Dock_Remaining"].iloc[0]

    # Skip plants with zero or missing dock capacity
    if dock_limit > 0:

        solver.Add(
            solver.Sum(x[i] for i in plant_orders) <= dock_limit
        )

        dock_constraints += 1

print("Dock Capacity Constraints Added Successfully!")
print("Plants with Dock Constraints:", dock_constraints)

Dock Capacity Constraints Added Successfully!
Plants with Dock Constraints: 6


In [15]:
# ==========================================================
# CELL 15 - Throughput Constraints
# ==========================================================

throughput_constraints = 0

for plant in sample_orders["Plant"].unique():

    plant_orders = sample_orders[
        sample_orders["Plant"] == plant
    ].index.tolist()

    throughput_limit = sample_orders[
        sample_orders["Plant"] == plant
    ]["Throughput_Capacity"].iloc[0]

    if throughput_limit > 0:

        solver.Add(
            solver.Sum(x[i] for i in plant_orders) <= throughput_limit
        )

        throughput_constraints += 1

print("Throughput Constraints Added Successfully!")
print("Plants with Throughput Constraints:", throughput_constraints)

Throughput Constraints Added Successfully!
Plants with Throughput Constraints: 8


In [16]:
# ==========================================================
# CELL 16 - Solve Optimization
# ==========================================================

# Set a time limit (5 seconds)
solver.SetTimeLimit(5000)

# Solve the optimization model
status = solver.Solve()

print("Solver Status:", status)

# Store selected orders
sample_orders["Selected"] = [
    int(x[i].solution_value())
    for i in sample_orders.index
]

# Calculate KPIs
selected_orders = sample_orders["Selected"].sum()
rejected_orders = len(sample_orders) - selected_orders

revenue = sample_orders.loc[
    sample_orders["Selected"] == 1,
    "Order_SKU_Revenue"
].sum()

fill_rate = (selected_orders / len(sample_orders)) * 100

print("\nOptimization Completed Successfully!")
print("Selected Orders :", selected_orders)
print("Rejected Orders :", rejected_orders)
print("Fill Rate       :", round(fill_rate, 2), "%")
print("Revenue         :", round(revenue, 2))

Solver Status: 0

Optimization Completed Successfully!
Selected Orders : 1369
Rejected Orders : 23824
Fill Rate       : 5.43 %
Revenue         : 27275767


In [17]:
# ==========================================================
# CELL 17 - Store Selected Orders
# ==========================================================

# Create a dataframe containing only selected orders
selected_orders_df = sample_orders[
    sample_orders["Selected"] == 1
].copy()

print("Selected Orders Dataset Created Successfully!")
print("Selected Orders Shape:", selected_orders_df.shape)

print("\nFirst 5 Selected Orders:")
print(selected_orders_df.head()) 

Selected Orders Dataset Created Successfully!
Selected Orders Shape: (1369, 41)

First 5 Selected Orders:
    Group_Flag Plant MaterialNumber transportationplanningdate IsTopCust  \
3   5484913123  5083        9517630                    6/26/24         N   
6   5484913123  5083        9519773                    6/26/24         N   
7   5484913123  5083       12575290                    6/26/24         N   
9   5484913123  5083       12584396                    6/26/24         N   
10  5484913123  5083       12154829                    6/26/24         N   

    OpeningStock RequestedDeliveryDate DeliveryNoteFlag IsInvAvail  \
3        17667.0               6/27/24                N          Y   
6        11272.0               6/27/24                N          Y   
7        73454.0               6/27/24                N          Y   
9        48222.0               6/27/24                N          Y   
10           0.0               6/27/24                N          N   

    LoadNumber  

In [18]:
# ==========================================================
# CELL 18 - Generate KPIs
# ==========================================================

total_orders = len(sample_orders)

selected_orders = sample_orders["Selected"].sum()

rejected_orders = total_orders - selected_orders

fill_rate = (selected_orders / total_orders) * 100

total_revenue = sample_orders.loc[
    sample_orders["Selected"] == 1,
    "Order_SKU_Revenue"
].sum()

total_shipping_cost = sample_orders.loc[
    sample_orders["Selected"] == 1,
    "Shipping_Cost"
].sum()

print("========== OPTIMIZATION KPIs ==========")
print(f"Total Orders        : {total_orders}")
print(f"Selected Orders     : {selected_orders}")
print(f"Rejected Orders     : {rejected_orders}")
print(f"Fill Rate (%)       : {fill_rate:.2f}")
print(f"Total Revenue       : {total_revenue:.2f}")
print(f"Total Shipping Cost : {total_shipping_cost:.2f}")

========== OPTIMIZATION KPIs ==========
Total Orders        : 25193
Selected Orders     : 1369
Rejected Orders     : 23824
Fill Rate (%)       : 5.43
Total Revenue       : 27275767.00
Total Shipping Cost : 3043034.45


In [19]:
# ==========================================================
# CELL 19 - Explainable Recommendations
# ==========================================================

sample_orders["Recommendation"] = np.where(
    sample_orders["Selected"] == 1,
    "Accept Order",
    "Reject Order"
)

sample_orders["Reason"] = np.where(
    sample_orders["Selected"] == 1,
    "High revenue and satisfies all optimization constraints",
    "Rejected due to inventory/capacity/dock/throughput or low profit"
)

print("Explainable Recommendations Generated Successfully!")

print(
    sample_orders[
        ["Selected", "Recommendation", "Reason"]
    ].head(10)
)

Explainable Recommendations Generated Successfully!
   Selected Recommendation                                             Reason
0         0   Reject Order  Rejected due to inventory/capacity/dock/throug...
1         0   Reject Order  Rejected due to inventory/capacity/dock/throug...
2         0   Reject Order  Rejected due to inventory/capacity/dock/throug...
3         1   Accept Order  High revenue and satisfies all optimization co...
4         0   Reject Order  Rejected due to inventory/capacity/dock/throug...
5         0   Reject Order  Rejected due to inventory/capacity/dock/throug...
6         1   Accept Order  High revenue and satisfies all optimization co...
7         1   Accept Order  High revenue and satisfies all optimization co...
8         0   Reject Order  Rejected due to inventory/capacity/dock/throug...
9         1   Accept Order  High revenue and satisfies all optimization co...


In [20]:
# ==========================================================
# CELL 20 - Risk Score
# ==========================================================

sample_orders["Risk_Score"] = np.where(
    sample_orders["Selected"] == 1,
    "Low",
    np.where(
        sample_orders["Available_inventory"] < sample_orders["OrderedQty_converted"],
        "High",
        "Medium"
    )
)

print("Risk Scores Generated Successfully!")

print(
    sample_orders[
        ["Selected", "Available_inventory", "OrderedQty_converted", "Risk_Score"]
    ].head(10)
)

Risk Scores Generated Successfully!
   Selected  Available_inventory  OrderedQty_converted Risk_Score
0         0             941546.0                    24     Medium
1         0             269679.0                    10     Medium
2         0             234719.0                     4     Medium
3         1             842948.0                   114        Low
4         0            2428992.0                    59     Medium
5         0                120.0                     7     Medium
6         1             335025.0                    80        Low
7         1            2001299.0                   216        Low
8         0                  0.0                     9       High
9         1            2651498.0                   144        Low


In [22]:
# ==========================================================
# CELL 21 - Carbon Emission Estimate
# ==========================================================

# Simple estimate: CO2 is proportional to shipping cost
sample_orders["Estimated_CO2_kg"] = (
    sample_orders["Shipping_Cost"] * 0.05 * sample_orders["Selected"]
)

total_co2 = sample_orders["Estimated_CO2_kg"].sum()

print("Carbon Emission Estimates Generated Successfully!\n")

print(sample_orders[
    ["Shipping_Cost", "Selected", "Estimated_CO2_kg"]
].head())

print("\nTotal Estimated CO2 Emissions (kg):", round(total_co2, 2))

Carbon Emission Estimates Generated Successfully!

   Shipping_Cost  Selected  Estimated_CO2_kg
0    2076.747562         0          0.000000
1    2076.747562         0          0.000000
2    2076.747562         0          0.000000
3    2076.747562         1        103.837378
4    2076.747562         0          0.000000

Total Estimated CO2 Emissions (kg): 152151.72


In [23]:
# ==========================================================
# CELL 22 - Classical Results Summary
# ==========================================================

summary = pd.DataFrame({
    "Metric": [
        "Total Orders",
        "Selected Orders",
        "Rejected Orders",
        "Fill Rate (%)",
        "Total Revenue",
        "Total Shipping Cost",
        "Estimated CO2 (kg)"
    ],
    "Value": [
        len(sample_orders),
        sample_orders["Selected"].sum(),
        len(sample_orders) - sample_orders["Selected"].sum(),
        round((sample_orders["Selected"].sum() / len(sample_orders)) * 100, 2),
        round(
            sample_orders.loc[
                sample_orders["Selected"] == 1,
                "Order_SKU_Revenue"
            ].sum(),
            2
        ),
        round(
            sample_orders.loc[
                sample_orders["Selected"] == 1,
                "Shipping_Cost"
            ].sum(),
            2
        ),
        round(
            sample_orders["Estimated_CO2_kg"].sum(),
            2
        )
    ]
})

print("Classical Results Summary Generated Successfully!\n")
print(summary)

Classical Results Summary Generated Successfully!

                Metric        Value
0         Total Orders     25193.00
1      Selected Orders      1369.00
2      Rejected Orders     23824.00
3        Fill Rate (%)         5.43
4        Total Revenue  27275767.00
5  Total Shipping Cost   3043034.45
6   Estimated CO2 (kg)    152151.72


In [24]:
# ==========================================================
# CELL 23 - Quantum Placeholder / Comparison
# ==========================================================

comparison_results = pd.DataFrame({
    "Metric": [
        "Selected Orders",
        "Revenue",
        "Fill Rate (%)",
        "Shipping Cost",
        "Estimated CO2 (kg)"
    ],
    "Classical Optimization": [
        sample_orders["Selected"].sum(),
        round(sample_orders.loc[
            sample_orders["Selected"] == 1,
            "Order_SKU_Revenue"
        ].sum(), 2),
        round(
            sample_orders["Selected"].sum() / len(sample_orders) * 100,
            2
        ),
        round(sample_orders.loc[
            sample_orders["Selected"] == 1,
            "Shipping_Cost"
        ].sum(), 2),
        round(sample_orders["Estimated_CO2_kg"].sum(), 2)
    ],
    "Quantum Optimization": [
        "Future Work",
        "Future Work",
        "Future Work",
        "Future Work",
        "Future Work"
    ]
})

print("Quantum Comparison Table Created Successfully!\n")
print(comparison_results)

Quantum Comparison Table Created Successfully!

               Metric  Classical Optimization Quantum Optimization
0     Selected Orders                 1369.00          Future Work
1             Revenue             27275767.00          Future Work
2       Fill Rate (%)                    5.43          Future Work
3       Shipping Cost              3043034.45          Future Work
4  Estimated CO2 (kg)               152151.72          Future Work


In [25]:
# ==========================================================
# CELL 24 - Save Advanced Optimization Results
# ==========================================================

sample_orders.to_csv(
    "../data/advanced_optimization_results.csv",
    index=False
)

print("advanced_optimization_results.csv saved successfully!")

advanced_optimization_results.csv saved successfully!


In [26]:
# ==========================================================
# CELL 25 - Save Comparison Results
# ==========================================================

comparison_results.to_csv(
    "../data/comparison_results.csv",
    index=False
)

print("comparison_results.csv saved successfully!")

comparison_results.csv saved successfully!


In [27]:
# ==========================================================
# CELL 26 - Save Dashboard Summary
# ==========================================================

summary.to_csv(
    "../data/dashboard_summary.csv",
    index=False
)

print("dashboard_summary.csv saved successfully!")

print("\n🎉 ALL 26 CELLS EXECUTED SUCCESSFULLY!")
print("Your Advanced Nestlé Order Optimization Project is complete.")

dashboard_summary.csv saved successfully!

🎉 ALL 26 CELLS EXECUTED SUCCESSFULLY!
Your Advanced Nestlé Order Optimization Project is complete.


In [28]:
# ==========================================================
# CELL 27 - Default Assignment
# ==========================================================

default_assignment = sample_orders.copy()

# Every order is assigned to its original plant
default_assignment["Assigned_Plant"] = default_assignment["Plant"]

# All orders are considered assigned
default_assignment["Default_Assigned"] = 1

# KPIs
default_total_orders = len(default_assignment)
default_selected_orders = default_assignment["Default_Assigned"].sum()
default_fill_rate = round(
    (default_selected_orders / default_total_orders) * 100,
    2
)

default_revenue = default_assignment["Order_SKU_Revenue"].sum()

default_shipping = default_assignment["Shipping_Cost"].sum()

print("====================================")
print("DEFAULT ASSIGNMENT RESULTS")
print("====================================")
print("Total Orders      :", default_total_orders)
print("Assigned Orders   :", default_selected_orders)
print("Fill Rate (%)     :", default_fill_rate)
print("Total Revenue     :", round(default_revenue, 2))
print("Total Shipping    :", round(default_shipping, 2))

print("\nFirst 5 Assignments:")
print(
    default_assignment[
        ["Plant", "Assigned_Plant", "Default_Assigned"]
    ].head()
)

DEFAULT ASSIGNMENT RESULTS
Total Orders      : 25193
Assigned Orders   : 25193
Fill Rate (%)     : 100.0
Total Revenue     : 88491861
Total Shipping    : 62172792.41

First 5 Assignments:
  Plant Assigned_Plant  Default_Assigned
0  5083           5083                 1
1  5083           5083                 1
2  5083           5083                 1
3  5083           5083                 1
4  5083           5083                 1


In [29]:
# ==========================================================
# CELL 28 - Greedy Assignment
# ==========================================================

greedy_assignment = sample_orders.copy()

# Sort orders by revenue (highest first)
greedy_assignment = greedy_assignment.sort_values(
    by="Order_SKU_Revenue",
    ascending=False
).reset_index(drop=True)

# Remaining inventory for each Plant-Material combination
remaining_inventory = (
    greedy_assignment
    .groupby(["Plant", "MaterialNumber"])["Available_inventory"]
    .first()
    .to_dict()
)

# Initialize assignment column
greedy_assignment["Greedy_Selected"] = 0

# Select orders greedily
for i in greedy_assignment.index:

    plant = greedy_assignment.loc[i, "Plant"]
    material = greedy_assignment.loc[i, "MaterialNumber"]
    qty = greedy_assignment.loc[i, "OrderedQty_converted"]

    key = (plant, material)

    if remaining_inventory.get(key, 0) >= qty:
        greedy_assignment.loc[i, "Greedy_Selected"] = 1
        remaining_inventory[key] -= qty

# KPIs
greedy_total_orders = len(greedy_assignment)
greedy_selected_orders = int(greedy_assignment["Greedy_Selected"].sum())
greedy_rejected_orders = greedy_total_orders - greedy_selected_orders

greedy_fill_rate = round(
    (greedy_selected_orders / greedy_total_orders) * 100,
    2
)

greedy_revenue = greedy_assignment.loc[
    greedy_assignment["Greedy_Selected"] == 1,
    "Order_SKU_Revenue"
].sum()

greedy_shipping = greedy_assignment.loc[
    greedy_assignment["Greedy_Selected"] == 1,
    "Shipping_Cost"
].sum()

print("====================================")
print("GREEDY ASSIGNMENT RESULTS")
print("====================================")
print("Total Orders      :", greedy_total_orders)
print("Selected Orders   :", greedy_selected_orders)
print("Rejected Orders   :", greedy_rejected_orders)
print("Fill Rate (%)     :", greedy_fill_rate)
print("Total Revenue     :", round(greedy_revenue, 2))
print("Shipping Cost     :", round(greedy_shipping, 2))

print("\nFirst 10 Greedy Assignments:")
print(
    greedy_assignment[
        [
            "Plant",
            "MaterialNumber",
            "Order_SKU_Revenue",
            "Greedy_Selected"
        ]
    ].head(10)
)

GREEDY ASSIGNMENT RESULTS
Total Orders      : 25193
Selected Orders   : 24842
Rejected Orders   : 351
Fill Rate (%)     : 98.61
Total Revenue     : 87710667
Shipping Cost     : 61332274.52

First 10 Greedy Assignments:
  Plant MaterialNumber  Order_SKU_Revenue  Greedy_Selected
0  5620       12568056             356600                1
1  5620       12568056             356600                1
2  5620       12568056             356600                1
3  5420       12568056             356600                1
4  5420       12568056             356600                1
5  5420       12568056             356600                1
6  5620       12568056             356600                1
7  5385       12568056             240019                1
8  5620       12482618             238801                1
9  5620       12482617             238801                1


In [30]:
# ==========================================================
# CELL 29 - OR-Tools Optimization Summary
# ==========================================================

ortools_total_orders = len(sample_orders)

ortools_selected_orders = int(sample_orders["Selected"].sum())

ortools_rejected_orders = (
    ortools_total_orders - ortools_selected_orders
)

ortools_fill_rate = round(
    (ortools_selected_orders / ortools_total_orders) * 100,
    2
)

ortools_revenue = sample_orders.loc[
    sample_orders["Selected"] == 1,
    "Order_SKU_Revenue"
].sum()

ortools_shipping = sample_orders.loc[
    sample_orders["Selected"] == 1,
    "Shipping_Cost"
].sum()

print("====================================")
print("OR-TOOLS OPTIMIZATION SUMMARY")
print("====================================")
print("Total Orders      :", ortools_total_orders)
print("Selected Orders   :", ortools_selected_orders)
print("Rejected Orders   :", ortools_rejected_orders)
print("Fill Rate (%)     :", ortools_fill_rate)
print("Total Revenue     :", round(ortools_revenue, 2))
print("Shipping Cost     :", round(ortools_shipping, 2))

OR-TOOLS OPTIMIZATION SUMMARY
Total Orders      : 25193
Selected Orders   : 1369
Rejected Orders   : 23824
Fill Rate (%)     : 5.43
Total Revenue     : 27275767
Shipping Cost     : 3043034.45


In [31]:
# ==========================================================
# CELL 30 - QUBO Formulation
# ==========================================================

qubo = pd.DataFrame()

qubo["Order_ID"] = sample_orders.index
qubo["Revenue"] = sample_orders["Order_SKU_Revenue"]
qubo["Shipping_Cost"] = sample_orders["Shipping_Cost"]
qubo["Penalty"] = sample_orders["Penaltyforpotentialcuts"].fillna(0)

# Objective coefficient
qubo["QUBO_Coefficient"] = (
    qubo["Revenue"]
    - qubo["Shipping_Cost"]
    - qubo["Penalty"]
)

print("====================================")
print("QUBO FORMULATION CREATED")
print("====================================")

print("Total Binary Variables :", len(qubo))
print("First 10 QUBO Variables:\n")

print(qubo.head(10))

QUBO FORMULATION CREATED
Total Binary Variables : 25193
First 10 QUBO Variables:

   Order_ID  Revenue  Shipping_Cost  Penalty  QUBO_Coefficient
0         0     1150    2076.747562      0.0       -926.747562
1         1      478    2076.747562      0.0      -1598.747562
2         2      832    2076.747562      0.0      -1244.747562
3         3     7364    2076.747562      0.0       5287.252438
4         4     1756    2076.747562      0.0       -320.747562
5         5      782    2076.747562      0.0      -1294.747562
6         6     5168    2076.747562      0.0       3091.252438
7         7     6067    2076.747562      0.0       3990.252438
8         8      729    2076.747562      0.0      -1347.747562
9         9     5502    2076.747562      0.0       3425.252438


In [32]:
# ==========================================================
# CELL 31 - QAOA (Quantum Optimization Placeholder)
# ==========================================================

qaoa_results = pd.DataFrame({
    "Metric": [
        "Solver",
        "Problem Type",
        "Binary Variables",
        "Quantum Status",
        "Backend",
        "Future Scope"
    ],
    "Value": [
        "QAOA",
        "QUBO",
        len(qubo),
        "Placeholder (Not Executed)",
        "IBM Quantum / Qiskit",
        "Run QAOA on real quantum hardware or simulator"
    ]
})

print("====================================")
print("QAOA PLACEHOLDER CREATED")
print("====================================")

print(qaoa_results)

QAOA PLACEHOLDER CREATED
             Metric                                           Value
0            Solver                                            QAOA
1      Problem Type                                            QUBO
2  Binary Variables                                           25193
3    Quantum Status                      Placeholder (Not Executed)
4           Backend                            IBM Quantum / Qiskit
5      Future Scope  Run QAOA on real quantum hardware or simulator


In [34]:
import os

print("Current Working Directory:")
print(os.getcwd())

Current Working Directory:
c:\Users\Bushra Faizi\OneDrive\Desktop\project\Nestle_DOM_Optimization\notebooks


In [36]:
# ==========================================================
# CELL 32 - Classical vs Quantum Comparison
# ==========================================================

phase2_comparison = pd.DataFrame({
    "Method": [
        "Default Assignment",
        "Greedy Assignment",
        "OR-Tools Optimization",
        "QAOA (Placeholder)"
    ],
    "Selected Orders": [
        default_selected_orders,
        greedy_selected_orders,
        ortools_selected_orders,
        "Future Work"
    ],
    "Fill Rate (%)": [
        default_fill_rate,
        greedy_fill_rate,
        ortools_fill_rate,
        "Future Work"
    ],
    "Revenue": [
        default_revenue,
        greedy_revenue,
        ortools_revenue,
        "Future Work"
    ],
    "Shipping Cost": [
        default_shipping,
        greedy_shipping,
        ortools_shipping,
        "Future Work"
    ]
})

print("=" * 60)
print("PHASE 2 - OPTIMIZATION METHODS COMPARISON")
print("=" * 60)
print(phase2_comparison)

# Save the comparison
phase2_comparison.to_csv(
    "../data/phase2_optimization_comparison.csv",
    index=False
)

print("✅ phase2_optimization_comparison.csv saved successfully!")



PHASE 2 - OPTIMIZATION METHODS COMPARISON
                  Method Selected Orders Fill Rate (%)      Revenue  \
0     Default Assignment           25193         100.0     88491861   
1      Greedy Assignment           24842         98.61     87710667   
2  OR-Tools Optimization            1369          5.43     27275767   
3     QAOA (Placeholder)     Future Work   Future Work  Future Work   

     Shipping Cost  
0  62172792.408451  
1  61332274.520043  
2   3043034.453954  
3      Future Work  
✅ phase2_optimization_comparison.csv saved successfully!
